<a href="https://colab.research.google.com/github/msklv/harbor-demo/blob/main/aiconf2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Поисковый агент за 45 минут
### AiConf · 20 апреля 2026 · Воркшоп

**Что вы построите:** Рабочий поисковый агент (ReAct loop) на основе Groq API + Tavily Search API — ~80 строк Python, без фреймворков, всё в браузере.

**Стек:** Python · Groq API (llama-3.3-70b) · Tavily Search API · Google Colab



## Шаг 1 — Установка зависимостей

Устанавливаем две библиотеки. Всё остальное уже есть в Colab.

In [ ]:
# Ячейка 1 — Установка зависимостей
# groq — SDK для Groq API (inference Llama 3.3 со скоростью 500 tok/s)
# requests — уже встроен в Python, но явно указываем
!pip install groq requests -q
print('[+] Зависимости установлены')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.5 MB/s eta 0:00:00
[+] Зависимости установлены


## Шаг 2 — Импорты

In [ ]:
# Ячейка 2 — Импорты
import os
import json
import requests
from groq import Groq
from getpass import getpass

print('[+] Импорты готовы')

[+] Импорты готовы


## Шаг 3 — API-ключи

Получить бесплатно:
- **Groq API**: https://console.groq.com → Sign Up → API Keys
- **Tavily API**: https://app.tavily.com → Sign Up → API Keys

> ⚠️ Ключи вводим через `getpass()` — они не отображаются на экране и не сохраняются в истории ячеек.

In [ ]:
# Ячейка 3 — API-ключи (вводим безопасно через getpass)
GROQ_API_KEY   = getpass('Введите Groq API key: ')
TAVILY_API_KEY = getpass('Введите Tavily API key: ')
print('[+] Ключи получены')

## Шаг 4 — Инструмент поиска ("руки" агента)

Функция `search()` — это то, что агент будет вызывать когда ему нужна актуальная информация.

**Tavily Search API** возвращает структурированные результаты из интернета — заголовок, контент, и синтезированный ответ.

In [ ]:
# Ячейка 4 — Search Tool
def search(query: str, max_results: int = 3) -> str:
    """
    Поиск в интернете через Tavily API.
    Возвращает структурированный текст с результатами.
    """
    response = requests.post(
        'https://api.tavily.com/search',
        json={
            'api_key': TAVILY_API_KEY,
            'query': query,
            'max_results': max_results,
            'include_answer': True,
        }
    )
    response.raise_for_status()
    data = response.json()

    results = []
    if data.get('answer'):
        results.append('NOTE: Quick answer: ' + str(data['answer']))
    for r in data.get('results', []):
        results.append('**' + r['title'] + '**\n' + r['content'][:300] + '...')

    return '\n\n'.join(results)

# Проверка: работает ли поиск?
print(search('что такое ReAct агент')[:500])

NOTE: Quick answer: A ReAct agent is an AI that combines reasoning and action, using a loop of thoughts and actions to solve complex tasks. It alternates between thinking and taking actions based on its environment. This framework helps AIs handle multi-step problems efficiently.

**What Are ReAct Agents?**
# ReAct Agent: The Ultimate Guide to the Reason and Act Framework for LLMs. ReAct agents merge reasoning with action, enabling AI to articulate thoughts and use tools to solve complex, multi-


## Шаг 5 — JSON-схема инструмента для LLM

LLM не умеет читать Python-код. Мы описываем инструмент в JSON-формате — и модель сама решает когда и с каким аргументом его вызвать.

Это называется **function calling** — модель возвращает не текст, а структурированный вызов функции.

In [ ]:
# Ячейка 5 — Tool Definition
tools = [{
    'type': 'function',
    'function': {
        'name': 'search',
        'description': 'Search the internet for current, up-to-date information. Use this for any factual question.',
        'parameters': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'The search query. Be specific and concise.'
                }
            },
            'required': ['query']
        }
    }
}]
print('[+] Tool definition готов')
print('Схема передаётся LLM — она сама решит когда вызывать search()')

[+] Tool definition готов
Схема передаётся LLM — она сама решит когда вызывать search()


## Шаг 6 — ReAct Loop (сердце агента)

**ReAct = Reasoning + Acting**

Паттерн: `подумал → поискал → посмотрел результат → снова подумал`

Цикл крутится пока LLM не скажет «готов ответить» (нет `tool_calls`) вместо «хочу искать» (есть `tool_calls`).

```
User Query
    ↓
[THINK]   LLM решает: нужен ли поиск?
    ↓ да
[ACT]     вызов search('запрос')
    ↓
[OBSERVE] получает результаты
    ↓
[THINK]   достаточно информации?
    ↓ нет → ACT (повтор)
    ↓ да
[ANSWER]  финальный ответ
```


In [ ]:
# Ячейка 6 — ReAct агент (~60 строк)
client = Groq(api_key=GROQ_API_KEY)

def run_agent(question, domain='general', max_iterations=5):
    """
    ReAct агент: LLM + поиск в интернете.

    Args:
        question: вопрос пользователя
        domain: специализация агента (влияет на system prompt)
        max_iterations: максимум итераций поиска (защита от зависания)

    Returns:
        финальный ответ агента (str)
    """
    system_prompt = (
        'You are a research agent specializing in: ' + domain + '.\n'
        'Rules:\n'
        '1. ALWAYS search before answering — never rely on training data alone.\n'
        '2. Search multiple times with different queries to get comprehensive info.\n'
        '3. Synthesize findings into a structured answer.\n'
        '4. Answer in the same language as the question.\n'
        '5. Be honest about what you found vs. what you inferred.'
    )

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': question}
    ]

    print('=> Вопрос: ' + question)
    print('   Домен: ' + domain)
    print()

    for i in range(max_iterations):
        print('[loop] Итерация ' + str(i+1) + '...')

        response = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=messages,
            tools=tools,
            tool_choice='auto',
            temperature=0.1
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            # Агент хочет искать — добавляем его вызов в историю
            messages.append({
                'role': 'assistant',
                'content': msg.content,
                'tool_calls': [
                    {
                        'id': tc.id,
                        'type': 'function',
                        'function': {
                            'name': tc.function.name,
                            'arguments': tc.function.arguments
                        }
                    }
                    for tc in msg.tool_calls
                ]
            })
            # Выполняем каждый вызов и добавляем результат в историю
            for tc in msg.tool_calls:
                query = json.loads(tc.function.arguments)['query']
                print('  >> Ищет: "' + query + '"')
                result = search(query)
                print('  .. получено ' + str(len(result)) + ' символов')
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tc.id,
                    'content': result
                })
            print()
        else:
            # Нет tool_calls → агент готов ответить
            print()
            print('[+] ОТВЕТ:')
            print('=' * 60)
            print(msg.content)
            print('=' * 60)
            print(':: Итераций поиска: ' + str(i + 1))
            return msg.content

    return '(!!) Достигнут лимит итераций (' + str(max_iterations) + ')'

print('[+] Агент готов!')

[+] Агент готов!


## Шаг 7 — ЗАПУСК! (тема выбрана аудиторией)

> 🗳️ **LIVE:** Открыть результаты Mentimeter. Вписать победившую тему вместо `[ПОБЕДИВШАЯ ТЕМА]`.

Варианты тем из голосования:
- Тренды ИИ в разработке 2025–2026
- Топ open-source инструментов для агентов
- Как конкуренты используют ИИ-поиск в продуктах


In [ ]:
# Ячейка 7 — ЗАПУСК! Тема выбрана аудиторией
# ↓ LIVE: вписать победившую тему прямо на сцене ↓
TOPIC = 'Тренды в ИИ в 2026 году'

result = run_agent(
    question=TOPIC,
    domain='AI и разработка',
    max_iterations=4
)

=> Вопрос: Тренды в ИИ в 2026 году
   Домен: AI и разработка

[loop] Итерация 1...
  >> Ищет: "ИИ тренды 2026"
  .. получено 1193 символов
  >> Ищет: "Искусственный интеллект тенденции 2026"
  .. получено 828 символов
  >> Ищет: "Технологии ИИ 2026"
  .. получено 958 символов

[loop] Итерация 2...

[+] ОТВЕТ:
В 2026 году ИИ интегрируется в умную автоматизацию и квантовые технологии. Тренды включают агентный ИИ и голосовые интерфейсы. Российские компании активно развивают собственные ИИ-решения. Основные тренды включают повседневное использование агентских технологий и решение кризисных ситуаций.
:: Итераций поиска: 2


---
## Блок 4: Ломаем агента



In [ ]:
# Ячейка 8 — Вопрос от аудитории
hard_question = 'Как собрать собственного агента для работы?'

run_agent(
    hard_question,
    domain='Агенты AI',
    max_iterations=5
)

=> Вопрос: Как собрать собственного агента для работы?
   Домен: Агенты AI

[loop] Итерация 1...
  >> Ищет: "создание агента AI с нуля"
  .. получено 888 символов
  >> Ищет: "требования к созданию агента AI"
  .. получено 813 символов
  >> Ищет: "технологии для создания агента AI"
  .. получено 1047 символов

[loop] Итерация 2...

[+] ОТВЕТ:
Для создания собственного агента AI необходимо определить цель, разработать архитектуру, и обеспечить обучение на данных. Использование нейросетей и базы знаний является критически важным. Можно использовать платформы как SiliconFlow или UiPath, и инструменты как Google ADK и Claude Agent SDK. Также важно обеспечить структурированное руководство и системы памяти.
:: Итераций поиска: 2


'Для создания собственного агента AI необходимо определить цель, разработать архитектуру, и обеспечить обучение на данных. Использование нейросетей и базы знаний является критически важным. Можно использовать платформы как SiliconFlow или UiPath, и инструменты как Google ADK и Claude Agent SDK. Также важно обеспечить структурированное руководство и системы памяти.'

### 3 типичных проблемы и их фиксы

| Проблема | Почему возникает | Fix |
|---|---|---|
| Галлюцинация — уверенный ответ без поиска | LLM «решила» что знает ответ | Добавить в промпт: "Always search before answering" |
| Бесконечный цикл поиска | Агент не может найти нужное | `max_iterations` (уже есть) + `timeout` на requests |
| Дорого в проде — много API-вызовов | Каждый поиск стоит денег | Кеширование по hash запроса |


In [ ]:
# Ячейка 9 — Fix 3: Кеширование поисковых запросов
import hashlib
import shelve

def search_cached(query: str) -> str:
    """
    search() с кешированием.
    Одинаковые запросы не идут в API повторно.
    """
    key = hashlib.md5(query.encode()).hexdigest()
    with shelve.open('search_cache') as cache:
        if key in cache:
            print('  [cache] Из кеша: "' + query + '"')
            return cache[key]
        result = search(query)
        cache[key] = result
        return result

print('[+] search_cached готов')
print('Теперь можно заменить search() на search_cached() в run_agent()')

[+] search_cached готов
Теперь можно заменить search() на search_cached() в run_agent()


---
## Блок 5: Ретроспектива и путь в прод


In [ ]:
# Ячейка 10 — Чеклист «Агент в прод»
print('''
╔══════════════════════════════════════════════════╗
║          ЧЕКЛИСТ: АГЕНТ В ПРОД                   ║
╠══════════════════════════════════════════════════╣
║  БАЗОВЫЙ ПРОТОТИП (у вас уже есть [+]):          ║
║    [+] ReAct loop с инструментом поиска          ║
║    [+] Специализация через system prompt         ║
║    [+] Ограничение итераций (max_iterations)     ║
╠══════════════════════════════════════════════════╣
║  ДО ПЕРВОГО ДЕПЛОЯ:                              ║
║    [ ] Структурированный вывод (Pydantic)        ║
║    [ ] Retry при ошибках API (tenacity)          ║
║    [ ] Кеширование запросов (Redis/shelve)       ║
║    [ ] Логирование каждого шага (Langfuse)       ║
║    [ ] Оценка качества ответов (eval dataset)    ║
╠══════════════════════════════════════════════════╣
║  ДЛЯ МАСШТАБИРОВАНИЯ:                            ║
║    [ ] Async (asyncio — параллельные поиски)     ║
║    [ ] Rate limiting (не спамить в API)          ║
║    [ ] Память (история + контекст юзера)         ║
║    [ ] Несколько инструментов (не только поиск) ║
║    [ ] Human-in-the-loop для критичных решений   ║
╚══════════════════════════════════════════════════╝
''')